In [58]:
from pymongo import MongoClient
import os
import javalang
from typing import List
import shutil
import json
# Load environment
from dotenv import load_dotenv
# OpenAPI model
import openai
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import Language
from langchain_openai import OpenAIEmbeddings  
from langchain.chains import RetrievalQA
from langchain.vectorstores.base import VectorStoreRetriever
# Embedding model
from langchain.document_loaders import TextLoader
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)
from langchain.vectorstores import FAISS
from langchain_core.language_models import BaseLanguageModel
from langchain.schema import Document
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda 

In [59]:
def retrieve_code_snippets(query_file_path: str, faiss_index_path: str, top_k: int = 2):
    # Đọc truy vấn từ file
    with open(query_file_path, 'r', encoding='utf-8') as f:
        query = f.read().strip()

    # Load FAISS index và tạo retriever
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.load_local(
        folder_path=faiss_index_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )

    # Truy xuất tài liệu
    retrieved_docs = vectorstore.similarity_search(query, k=top_k)

    return query, retrieved_docs
# Retriver thủ công (list docs)
class StaticRetriever:
    def __init__(self, documents: list[Document]):
        self.documents = documents

    def get_relevant_documents(self, query: str) -> list[Document]:
        return self.documents

def answer_with_llm(llm: BaseLanguageModel, query: str, retrieved_docs: list[Document]):
    # Tạo retriever thường
    retriever = StaticRetriever(retrieved_docs)

    # 🆕 Wrap retriever thành Runnable để tương thích với LangChain mới
    runnable_retriever = RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"]))

    # Prompt RAG đơn giản
    prompt = ChatPromptTemplate.from_template("""
    Use the context to answer the question. If you don't know, say you don't know.

    Context:
    {context}

    Question:
    {input}
    """)

    # Tạo chain nhồi context
    combine_docs_chain = create_stuff_documents_chain(llm, prompt)

    # Tạo full retrieval chain
    rag_chain = create_retrieval_chain(
        retriever=runnable_retriever,
        combine_docs_chain=combine_docs_chain
    )

    # Gọi LLM
    result = rag_chain.invoke({"input": query})

    return {
        "query": query,
        "answer": result["answer"],
        "context_snippets": [
            {
                "content": doc.page_content,
                "metadata": doc.metadata
            }
            for doc in retrieved_docs
        ]
    }

In [60]:
vectorDB_directory = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-rag\vectorDB"

In [61]:
# Setup model
# Load environment variables
load_dotenv()

# Retrieve API key
api_key = os.getenv("OPENAI_API_KEY")
# Ensure the API key is correctly set
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables")

# Initialize the ChatOpenAI model
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    openai_api_key=api_key  # Ensure you explicitly pass the API key
)

In [62]:
# Step 1: Retrieval FAISS
query, docs = retrieve_code_snippets("query.txt", vectorDB_directory)
print("query: ",query)
print("docs: ",docs)
# Step 2: Argumented
response = answer_with_llm(llm, query, docs)

# In ra kết quả

print(json.dumps(response, indent=2, ensure_ascii=False))

query:  CompilationUnit.ClassDeclaration -> {'public'}
CompilationUnit.ClassDeclaration -> TempClass
CompilationUnit.ClassDeclaration.MethodDeclaration -> {'public', 'static'}
CompilationUnit.ClassDeclaration.MethodDeclaration -> dummyMethod
CompilationUnit.ClassDeclaration.MethodDeclaration.StatementExpression.MethodInvocation -> AppLovinPrivacySettings
CompilationUnit.ClassDeclaration.MethodDeclaration.StatementExpression.MethodInvocation.Literal -> false
CompilationUnit.ClassDeclaration.MethodDeclaration.StatementExpression.MethodInvocation -> setHasUserConsent
CompilationUnit.ClassDeclaration.MethodDeclaration.StatementExpression.MethodInvocation -> AppLovinPrivacySettings
CompilationUnit.ClassDeclaration.MethodDeclaration.StatementExpression.MethodInvocation.Literal -> true
CompilationUnit.ClassDeclaration.MethodDeclaration.StatementExpression.MethodInvocation -> setDoNotSell
docs:  [Document(id='afd654b4-30e1-48be-b361-925500fddbdd', metadata={'source': 'C:\\Users\\ASUS\\anaconda